<a href="https://colab.research.google.com/github/Priyankapawar1224/3A-Handling-Numerical-Data-3B-Practical-Handling-Categorial-Data-/blob/main/Practical_3Ab_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
df = pd.read_csv("/content/Student Mental health (1).csv")
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True).str.lower().str.replace(' ', '_')
print("--- Initial Dataset Info ---")
print(df.info())
print("\n--- First 5 Rows ---")
print(df.head())

--- Initial Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   timestamp                             101 non-null    object 
 1   chooseyourgender                      101 non-null    object 
 2   age                                   100 non-null    float64
 3   whatisyourcourse                      101 non-null    object 
 4   yourcurrentyearofstudy                101 non-null    object 
 5   whatisyourcgpa                        101 non-null    object 
 6   maritalstatus                         101 non-null    object 
 7   doyouhavedepression                   101 non-null    object 
 8   doyouhaveanxiety                      101 non-null    object 
 9   doyouhavepanicattack                  101 non-null    object 
 10  didyouseekanyspecialistforatreatment  101 non-null    obj

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, Normalizer
df = pd.read_csv("/content/Student Mental health (1).csv")
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True).str.lower().str.replace(' ', '_')
ender']].copy()

df_for_norm.dropna(inplace=True)
df_for_norm = pd.get_dummies(df_for_norm, columns=['chooseyourgender'], drop_first=True)
df_for_norm.rename(columns={'chooseyourgender_Male': 'is_male'}, inplace=True)
standard_scaler_norm = StandardScaler()
df_for_norm['age_standardized'] = standard_scaler_norm.fit_transform(df_for_norm[['age']])
features_to_normalize = df_for_norm[['age_standardized', 'is_male']]

normalizer = Normalizer()
normalized_data = normalizer.fit_transform(features_to_normalize)

df_for_norm[['age_norm', 'is_male_norm']] = normalized_data

print("--- Normalized Observations (L2-Norm) Results ---")
print(df_for_norm[['age_standardized', 'is_male', 'age_norm', 'is_male_norm']].head())
print("\nVerification (Sum of squares of normalized values in the first row):")
# The sum of squares of normalized values for each row should be 1
print(f"Row 1: {df_for_norm.iloc[0]['age_norm']**2 + df_for_norm.iloc[0]['is_male_norm']**2 :.2f}")

--- Normalized Observations (L2-Norm) Results ---
   age_standardized  is_male  age_norm  is_male_norm
0         -1.018614    False -1.000000      0.000000
1          0.189229     True  0.185929      0.982563
2         -0.616000     True -0.524477      0.851424
3          0.591843    False  1.000000      0.000000
4          0.994457     True  0.705139      0.709069

Verification (Sum of squares of normalized values in the first row):
Row 1: 1.00


In [ ]:

df_miss = df.copy()
missing_indices = np.random.choice(df_miss.index, 5, replace=False)
df_miss.loc[missing_indices, 'age'] = np.nan

print("\n--- Deleting Observations with Missing Values ---")
print(f"Dataset size before deletion: {len(df_miss)}")

# Use dropna() to delete rows with ANY missing value
df_deleted = df_miss.dropna()

print(f"Dataset size after deleting rows with missing 'age' values: {len(df_deleted)}")


--- Deleting Observations with Missing Values ---
Dataset size before deletion: 101
Dataset size after deleting rows with missing 'age' values: 95


In [ ]:
print("\n--- Imputing Missing Values (Median Imputation) ---")
missing_count = df_miss['age'].isna().sum()
print(f"Number of missing 'age' values before imputation: {missing_count}")
median_imputer = SimpleImputer(strategy='median')
df_miss['age_imputed'] = median_imputer.fit_transform(df_miss[['age']])
# Check the imputed values where the original was NaN
imputed_check = df_miss[df_miss['age'].isna()][['age', 'age_imputed']].head()
print(f"The Median used for imputation is: {df_miss['age'].median():.2f}")
print("Imputed rows sample:")
print(imputed_check)
print(f"\nNumber of missing 'age_imputed' values after imputation: {df_miss['age_imputed'].isna().sum()}")


--- Imputing Missing Values (Median Imputation) ---
Number of missing 'age' values before imputation: 6
The Median used for imputation is: 19.00
Imputed rows sample:
    age  age_imputed
43  NaN         19.0
45  NaN         19.0
55  NaN         19.0
66  NaN         19.0
67  NaN         19.0

Number of missing 'age_imputed' values after imputation: 0


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
df = pd.read_csv("/content/Student Mental health (1).csv")
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True).str.lower().str.replace(' ', '_')
df_cluster = df.copy()
def clean_year(year_str):
    if isinstance(year_str, str):
        year_str = year_str.lower().replace(' ', '').replace('.', '') # Added replace('.', '') for robust cleaning
        if 'year1' in year_str: return 1
        elif 'year2' in year_str: return 2
        elif 'year3' in year_str: return 3
        elif 'year4' in year_str: return 4
    return np.nan # For any unmatchable values
df_cluster['study_year_num'] = df_cluster['yourcurrentyearofstudy'].apply(clean_year)

# 2. Select and Standardize the features
features_for_clustering = df_cluster[['age', 'study_year_num']].dropna()

# Standardize the features (mandatory for distance-based algorithms like K-Means)
scaler_kmeans = StandardScaler()
X_scaled = scaler_kmeans.fit_transform(features_for_clustering)

# 3. Apply K-Means Clustering (let's assume K=3 for this example)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
features_for_clustering['cluster'] = kmeans.fit_predict(X_scaled)

print("\n--- Corrected Grouping Observations using K-Means Clustering (K=3) ---")
print("Cluster Assignment for the first 10 observations:")
print(features_for_clustering[['age', 'study_year_num', 'cluster']].head(10))

print("\nSummary of Clusters (Mean Age and Study Year):")
# Show the mean age and study year for each cluster to understand the groups
cluster_summary = features_for_clustering.groupby('cluster')[['age', 'study_year_num']].mean()
print(cluster_summary)


--- Corrected Grouping Observations using K-Means Clustering (K=3) ---
Cluster Assignment for the first 10 observations:
    age  study_year_num  cluster
0  18.0               1        1
1  21.0               2        2
2  19.0               1        1
3  22.0               3        0
4  23.0               4        0
5  19.0               2        1
6  23.0               2        2
7  18.0               1        1
8  19.0               2        1
9  18.0               1        1

Summary of Clusters (Mean Age and Study Year):
               age  study_year_num
cluster                           
0        21.406250        3.250000
1        18.511111        1.377778
2        23.260870        1.391304


In [ ]:

data_rescale = df[['age']].copy()

# Initialize the MinMaxScaler
min_max_scaler = MinMaxScaler()

# Apply the scaling
data_rescale['age_rescaled'] = min_max_scaler.fit_transform(data_rescale[['age']])

print("--- Rescaling (Min-Max Scaling) on 'age' ---")
print(data_rescale[['age', 'age_rescaled']].head())
print(f"\nOriginal Min/Max Age: {df['age'].min()}/{df['age'].max()}")
print(f"Rescaled Min/Max Age: {data_rescale['age_rescaled'].min():.2f}/{data_rescale['age_rescaled'].max():.2f}")

--- Rescaling (Min-Max Scaling) on 'age' ---
    age  age_rescaled
0  18.0      0.000000
1  21.0      0.500000
2  19.0      0.166667
3  22.0      0.666667
4  23.0      0.833333

Original Min/Max Age: 18.0/24.0
Rescaled Min/Max Age: 0.00/1.00


In [ ]:

data_standardize = df[['age']].copy()

# Initialize the StandardScaler
standard_scaler = StandardScaler()

# Apply the standardization
data_standardize['age_standardized'] = standard_scaler.fit_transform(data_standardize[['age']])

print("\n--- Standardizing (Z-Score) on 'age' ---")
print(data_standardize[['age', 'age_standardized']].head())
print(f"\nOriginal Mean/Std Dev: {df['age'].mean():.2f}/{df['age'].std():.2f}")
print(f"Standardized Mean/Std Dev: {data_standardize['age_standardized'].mean():.2f}/{data_standardize['age_standardized'].std():.2f}")


--- Standardizing (Z-Score) on 'age' ---
    age  age_standardized
0  18.0         -1.018614
1  21.0          0.189229
2  19.0         -0.616000
3  22.0          0.591843
4  23.0          0.994457

Original Mean/Std Dev: 20.53/2.50
Standardized Mean/Std Dev: -0.00/1.01


In [ ]:
import pandas as pd
df = pd.read_csv("/content/Student Mental health (1).csv")
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True).str.lower().str.replace(' ', '_')

# --- Perform Feature Dummification (One-Hot Encoding) ---

# 1. Select the DataFrame to dummify. We drop 'timestamp' as it's not useful for OHE.
df_dummified = df.drop(columns=['timestamp']).copy()

# 2. Apply pd.get_dummies() to all remaining categorical (object) columns.
# drop_first=True is used to avoid multicollinearity by dropping one category per feature.
df_dummified = pd.get_dummies(
    df_dummified,
    drop_first=True,
    dummy_na=False
)

print(f"Original number of columns: {df.shape[1]}")
print(f"New number of columns after dummification: {df_dummified.shape[1]}")
print("\nFirst 5 rows of the dummified data:")
print(df_dummified.head())

Original number of columns: 11
New number of columns after dummification: 66

First 5 rows of the dummified data:
    age  chooseyourgender_Male  whatisyourcourse_Accounting   \
0  18.0                  False                         False   
1  21.0                   True                         False   
2  19.0                   True                         False   
3  22.0                  False                         False   
4  23.0                   True                         False   

   whatisyourcourse_BCS  whatisyourcourse_BENL  whatisyourcourse_BIT  \
0                 False                  False                 False   
1                 False                  False                 False   
2                 False                  False                  True   
3                 False                  False                 False   
4                 False                  False                 False   

   whatisyourcourse_Banking Studies  whatisyourcourse_Benl  \
0     

**Encoding Nominal Categorical Feature **

In [ ]:
feature_gender = df[['chooseyourgender']].values

# Create and fit the one-hot encoder
one_hot_lb = LabelBinarizer()
gender_encoded = one_hot_lb.fit_transform(feature_gender)

print("--- LabelBinarizer Output ---")
print(f"Encoded classes: {one_hot_lb.classes_}")
print(f"First 5 encoded values: {gender_encoded[:5].flatten()}")

--- LabelBinarizer Output ---
Encoded classes: ['Female' 'Male']
First 5 encoded values: [0 1 1 0 1]


In [ ]:

anxiety_dummies = pd.get_dummies(df['doyouhaveanxiety'], drop_first=True)

print("\n--- pandas.get_dummies Output (Anxiety) ---")
print(anxiety_dummies.head().rename(columns={'Yes': 'anxiety_yes'}))


--- pandas.get_dummies Output (Anxiety) ---
   anxiety_yes
0        False
1         True
2         True
3        False
4        False


Encoding Ordinal Categorical Features



In [ ]:
df_ordinal = df[['yourcurrentyearofstudy']].copy()
df_ordinal['yourcurrentyearofstudy'] = df_ordinal['yourcurrentyearofstudy'].str.lower().str.replace(' ', '').str.replace('.', '')

scale_mapper = {
    "year1": 1,
    "year2": 2,
    "year3": 3,
    "year4": 4
}
df_ordinal['study_year_encoded'] = df_ordinal['yourcurrentyearofstudy'].replace(scale_mapper)

print("\n--- Ordinal Encoding (Study Year) Output ---")
print(df_ordinal[['yourcurrentyearofstudy', 'study_year_encoded']].head())


--- Ordinal Encoding (Study Year) Output ---
  yourcurrentyearofstudy  study_year_encoded
0                  year1                   1
1                  year2                   2
2                  year1                   1
3                  year3                   3
4                  year4                   4


/tmp/ipython-input-3840915029.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_ordinal['study_year_encoded'] = df_ordinal['yourcurrentyearofstudy'].replace(scale_mapper)


In [ ]:
# Prepare data: Inject a NaN into 'maritalstatus' for demonstration
df_impute_cat = df[['maritalstatus']].copy()
np.random.seed(42)
missing_index = np.random.choice(df_impute_cat.index, 1)
df_impute_cat.loc[missing_index, 'maritalstatus'] = np.nan

# Create and fit the imputer with 'most_frequent' strategy
imputer_cat = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
imputed_data = imputer_cat.fit_transform(df_impute_cat.values)

mode_value = df_impute_cat['maritalstatus'].mode()[0]

print("\n--- Categorical Imputation Output ---")
print(f"Most Frequent Value (Mode): {mode_value}")
print(f"Imputed Feature (first 5): {imputed_data[:5].flatten()}")


--- Categorical Imputation Output ---
Most Frequent Value (Mode): No
Imputed Feature (first 5): ['No' 'No' 'No' 'Yes' 'No']


In [ ]:
# Create binary target (1 for Depression, 0 otherwise)
target = np.where(df['doyouhavedepression'] == 'Yes', 1, 0)

# Use 'age' as a simple feature set (impute NaNs first)
features = df['age'].fillna(df['age'].median()).values.reshape(-1, 1)
features = StandardScaler().fit_transform(features)

class_0_count = np.sum(target == 0) # No Depression
class_1_count = np.sum(target == 1) # Depression

print("\n--- Imbalanced Class Handling ---")
print(f"Original Class Balance: Class 0 (No): {class_0_count}, Class 1 (Yes): {class_1_count}")

clf_weighted = RandomForestClassifier(class_weight='balanced', random_state=42)

print(f"\nClassifier created with Class Weighting: {clf_weighted}")

i_class0 = np.where(target == 0)[0] # Indices of Class 0 (Majority)
i_class1 = np.where(target == 1)[0] # Indices of Class 1 (Minority)
n_class1 = len(i_class1) # Size of Minority Class
np.random.seed(42)
i_class0_downsampled = np.random.choice(i_class0, size=n_class1, replace=False)

i_downsampled = np.hstack((i_class0_downsampled, i_class1))

# Create the new balanced dataset
target_downsampled = target[i_downsampled]
features_downsampled = features[i_downsampled]

print("\n--- Downsampling Output ---")
print(f"New Target Balance: Class 0: {np.sum(target_downsampled == 0)}, Class 1: {np.sum(target_downsampled == 1)}")
print(f"Downsampled Feature Matrix Shape: {features_downsampled.shape}")


--- Imbalanced Class Handling ---
Original Class Balance: Class 0 (No): 66, Class 1 (Yes): 35

Classifier created with Class Weighting: RandomForestClassifier(class_weight='balanced', random_state=42)

--- Downsampling Output ---
New Target Balance: Class 0: 35, Class 1: 35
Downsampled Feature Matrix Shape: (70, 1)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
df = pd.read_csv("/content/Student Mental health (1).csv")
df.columns = df.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True).str.lower().str.replace(' ', '_')
def clean_year(year_str):
    if isinstance(year_str, str):
        year_str = year_str.lower().replace(' ', '').replace('.', '')
        if 'year1' in year_str: return 1
        elif 'year2' in year_str: return 2
        elif 'year3' in year_str: return 3
        elif 'year4' in year_str: return 4
    return np.nan

df['study_year_num'] = df['yourcurrentyearofstudy'].apply(clean_year)
data_knn = df[['age', 'study_year_num', 'maritalstatus']].copy()
data_knn.dropna(subset=['age', 'study_year_num'], inplace=True)
np.random.seed(42)
missing_indices = np.random.choice(data_knn.index, 5, replace=False)
data_knn.loc[missing_indices, 'maritalstatus'] = np.nan
df_train = data_knn.dropna(subset=['maritalstatus'])
df_missing = data_knn[data_knn['maritalstatus'].isna()]
X_train_raw = df_train[['age', 'study_year_num']].values
y_train_raw = df_train['maritalstatus'].values
X_missing_raw = df_missing[['age', 'study_year_num']].values

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_raw)

# Scale Features (Age and Study Year)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_missing_scaled = scaler.transform(X_missing_raw)
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_model.fit(X_train_scaled, y_train_encoded)

# Predict the missing values
y_predicted_encoded = knn_model.predict(X_missing_scaled)

# Convert predicted labels back to 'No'/'Yes'
y_predicted_labels = label_encoder.inverse_transform(y_predicted_encoded)

print("\n--- KNN Imputation Results ---")
imputation_results = pd.DataFrame({
    'Row Index': df_missing.index,
    'Predicted_Marital_Status': y_predicted_labels
})
print(imputation_results)


--- KNN Imputation Results ---
   Row Index Predicted_Marital_Status
0         45                       No
1         46                       No
2         54                       No
3         71                       No
4         84                       No
